# IMC Prosperity 4 — Round 5, 50-product research notebook

This notebook is built to run from a local folder called `DATA_DIR`, containing files like:

```text
prices_round_5_day_2.csv
prices_round_5_day_3.csv
prices_round_5_day_4.csv
```

It analyses:

1. product stability / volatility / spreads,
2. name-family sectors and data-driven clusters,
3. correlation and PCA structure,
4. synthetic family ETFs / baskets,
5. product-vs-basket residual spreads,
6. pair spreads inside each family,
7. simple z-score mean-reversion edge diagnostics.

## Main findings from the uploaded days 2–4

The 50 products naturally split into **10 families of 5 products**:

`GALAXY_SOUNDS`, `MICROCHIP`, `OXYGEN_SHAKE`, `PANEL`, `PEBBLES`, `ROBOT`, `SLEEP_POD`, `SNACKPACK`, `TRANSLATOR`, `UV_VISOR`.

The cleanest high-level observations:

- **SNACKPACK is the most stable family** by tick return volatility.
- **PEBBLES and MICROCHIP are the most volatile families**, so they may have bigger gross moves but need stricter risk control.
- Tick-to-tick return correlations are mostly weak. The obvious family split is useful for basket construction, but it is **not** strongly confirmed by raw return correlations.
- PCA does **not** show one dominant market factor: PC1 explains only about **5.6%** of standardized return variance, and the first 10 PCs explain about **28.3%**.
- The best first research direction is not a classical correlation ETF. It is more likely **residual mean reversion**: product vs family basket, or selected pair spreads.
- The strongest gross basket-residual candidates around a 200-tick horizon were: `ROBOT_DISHES`, `MICROCHIP_OVAL`, `ROBOT_LAUNDRY`, `ROBOT_MOPPING`, `SLEEP_POD_NYLON`, `ROBOT_VACUUMING`, `PEBBLES_S`, `UV_VISOR_AMBER`.
- The strongest gross pair-spread candidates around a 200-tick horizon were: `ROBOT_LAUNDRY vs ROBOT_VACUUMING`, `ROBOT_LAUNDRY vs ROBOT_MOPPING`, `PEBBLES_S vs PEBBLES_XL`, `ROBOT_DISHES vs ROBOT_IRONING`, `UV_VISOR_AMBER vs UV_VISOR_MAGENTA`.

Important: the edge diagnostics below are **gross** residual-reversion diagnostics. Before implementing, we need to check transaction costs, position limits, whether we can trade the basket legs cleanly, and whether the edge survives a proper day-by-day walk-forward backtest.

In [12]:
# =========================
# 0. Imports and config
# =========================
from pathlib import Path
import os, glob, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Change this if your folder has a different name/path.
DATA_DIR = Path("DATA_DIR")

# Fallback path if DATA_DIR is not found
if not DATA_DIR.exists():
    DATA_DIR = Path("/mnt/data")

print("Using DATA_DIR =", DATA_DIR.resolve())

Using DATA_DIR = DATA_DIR


In [13]:
# =========================
# 1. Load price files
# =========================
def load_round5_prices(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.glob("prices_round_5_day_*.csv"))
    if not files:
        files = sorted(data_dir.glob("*round_5*day*.csv"))
    if not files:
        raise FileNotFoundError(f"No Round 5 price files found in {data_dir}")

    frames = []
    for f in files:
        tmp = pd.read_csv(f, sep=";")
        tmp["source_file"] = f.name
        frames.append(tmp)

    df = pd.concat(frames, ignore_index=True)
    df = df.sort_values(["day", "timestamp", "product"]).reset_index(drop=True)
    return df

df = load_round5_prices(DATA_DIR)

print(df.shape)
print("Days:", sorted(df["day"].unique()))
print("Products:", df["product"].nunique())
display(df.head())

(1500000, 18)
Days: [np.int64(2), np.int64(3), np.int64(4)]
Products: 50


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,source_file
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,prices_round_5_day_2.csv
1,2,0,GALAXY_SOUNDS_DARK_MATTER,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,prices_round_5_day_2.csv
2,2,0,GALAXY_SOUNDS_PLANETARY_RINGS,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,prices_round_5_day_2.csv
3,2,0,GALAXY_SOUNDS_SOLAR_FLAMES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,prices_round_5_day_2.csv
4,2,0,GALAXY_SOUNDS_SOLAR_WINDS,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,prices_round_5_day_2.csv


In [14]:
# =========================
# 2. Infer families / sectors
# =========================
KNOWN_FAMILIES = [
    "GALAXY_SOUNDS", "MICROCHIP", "OXYGEN_SHAKE", "PANEL", "PEBBLES",
    "ROBOT", "SLEEP_POD", "SNACKPACK", "TRANSLATOR", "UV_VISOR"
]

def infer_family(product: str) -> str:
    for f in KNOWN_FAMILIES:
        if product.startswith(f + "_"):
            return f
    return product.rsplit("_", 1)[0]

products = sorted(df["product"].unique())
family_map = {p: infer_family(p) for p in products}

families_df = pd.DataFrame({
    "product": products,
    "family": [family_map[p] for p in products],
})
display(families_df.groupby("family")["product"].apply(list).to_frame("products"))

,products
family,
GALAXY_SOUNDS,"[GALAXY_SOUNDS_BLACK_HOLES, GALAXY_SOUNDS_DARK..."
MICROCHIP,"[MICROCHIP_CIRCLE, MICROCHIP_OVAL, MICROCHIP_R..."
OXYGEN_SHAKE,"[OXYGEN_SHAKE_CHOCOLATE, OXYGEN_SHAKE_EVENING_..."
PANEL,"[PANEL_1X2, PANEL_1X4, PANEL_2X2, PANEL_2X4, P..."
PEBBLES,"[PEBBLES_L, PEBBLES_M, PEBBLES_S, PEBBLES_XL, ..."
ROBOT,"[ROBOT_DISHES, ROBOT_IRONING, ROBOT_LAUNDRY, R..."
SLEEP_POD,"[SLEEP_POD_COTTON, SLEEP_POD_LAMB_WOOL, SLEEP_..."
SNACKPACK,"[SNACKPACK_CHOCOLATE, SNACKPACK_PISTACHIO, SNA..."
TRANSLATOR,"[TRANSLATOR_ASTRO_BLACK, TRANSLATOR_ECLIPSE_CH..."


## Pair spreads inside each family

For every pair in a family, this estimates:

\[
\log P_y(t) - \log P_y(0) = a + \beta \left(\log P_x(t)-\log P_x(0)\right) + \epsilon_t
\]

Then the residual is tested for mean reversion.

This is often easier to implement than a full basket because only two legs are involved, but you still need to check both products' spreads, limits, and fills.

## What I would test next for an actual strategy

1. Start with a **product-vs-family-basket fair value** for the strongest candidates:
   - `ROBOT_DISHES`
   - `ROBOT_LAUNDRY`
   - `ROBOT_MOPPING`
   - `ROBOT_VACUUMING`
   - `MICROCHIP_OVAL`
   - `PEBBLES_S`
   - `UV_VISOR_AMBER`

2. Also test pair spreads:
   - `ROBOT_LAUNDRY vs ROBOT_VACUUMING`
   - `ROBOT_LAUNDRY vs ROBOT_MOPPING`
   - `ROBOT_DISHES vs ROBOT_IRONING`
   - `PEBBLES_S vs PEBBLES_XL`
   - `UV_VISOR_AMBER vs UV_VISOR_MAGENTA`
   - `UV_VISOR_AMBER vs UV_VISOR_YELLOW`

3. For implementation, do not immediately trade every z-score crossing. Use:
   - entry threshold around `|z| >= 2`,
   - exit around `z = 0` or `|z| < 0.5`,
   - inventory cap per product,
   - quote-based/passive entries first,
   - aggressive taking only if the model edge is much larger than spread + safety margin.

4. Run day-by-day validation:
   - train coefficients on day 2, test day 3;
   - train days 2–3, test day 4;
   - only keep candidates that survive out-of-sample.

5. Be careful with basket implementation:
   - a full 5-leg basket is expensive and can create inventory problems;
   - a basket fair can still be useful even if the actual trade is only the mispriced product.